In [1]:
import pandas as pd
import numpy as np

In [2]:
customers = pd.read_csv("../Raw_Dataset/customers_dataset.csv")
orders = pd.read_csv("../Raw_Dataset/orders_dataset.csv")
order_items = pd.read_csv("../Raw_Dataset/order_items_dataset.csv")
products = pd.read_csv("../Raw_Dataset/products_dataset.csv")
payments = pd.read_csv("../Raw_Dataset/order_payments_dataset.csv")
category_translation = pd.read_csv("../Raw_Dataset/product_category_name_translation.csv")

In [3]:
customers.info()
orders.info()
order_items.info()
products.info()
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4

In [4]:
# checking missing values
orders.isnull().sum()
order_items.isnull().sum()
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [5]:
# Drop products without category (610 rows)
products = products.dropna(subset=['product_category_name'])

# Fill minor numeric missing values with median (2 rows)
products.fillna(products.median(numeric_only=True), inplace=True)

In [6]:
# convert date column
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

In [7]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
# removing cancel order
orders = orders[orders['order_status'] == 'delivered']

In [9]:
# Check shape
print("Total Delivered Orders:", orders.shape[0])
print("Total Order Items:", order_items.shape[0])

Total Delivered Orders: 96478
Total Order Items: 112650


In [10]:
# Keep only order_items that is to be delivered
order_items = order_items[order_items['order_id'].isin(orders['order_id'])]
print("Total order items which is delivered:", order_items.shape[0])

Total order items which is delivered: 110197


In [11]:
# merge category translation
products = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

In [12]:
# create revenue column (Revenue = price + freight)
order_items['total_price'] = order_items['price'] + order_items['freight_value']

In [13]:
# saving clean data
customers.to_csv("../Cleaned_Data/customers_clean.csv", index=False)
orders.to_csv("../Cleaned_Data/orders_clean.csv", index=False)
order_items.to_csv(
    "../Cleaned_Data/order_items_clean_pipe_utf16.csv",
    index=False,
    sep='|',
    encoding='utf-16'
)
products.to_csv(
    "../Cleaned_Data/products_clean_pipe_utf16.csv",
    index=False,
    sep='|',
    encoding='utf-16'
)
payments.to_csv(
    "../Cleaned_Data/payments_clean_pipe_utf16.csv",
    index=False,
    sep='|',
    encoding='utf-16'
)